# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a template for loading and exploring the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library. The dataset includes mass spectrometry measurements with detailed protein-level information and metadata.

### Dataset Source
The dataset source is provided via a Croissant schema URL and is fully referenced by `@id` for entities such as record sets and fields.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`. This step retrieves all schema-level information and prepares objects for programmatic access.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL (Croissant schema)
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset name: {metadata.name}\nDescription: {metadata.description}\nIdentifier: {metadata.identifier}\nVersion: {metadata.version}")

## 2. Data Overview

Review available record sets, fields, and their IDs. All entities are referenced by their `@id` as described in the Croissant specification.

**NOTE:** You can always view the full list of metadata entities programmatically.

In [ ]:
# List all available record sets, with their @id
record_sets = metadata.record_sets
print(f"This dataset contains {len(record_sets)} record set(s):\n")

for record_set in record_sets:
    print(f"- @id: {record_set['@id']}")
    print(f"  Name: {record_set.get('name', '(unnamed record set)')}")
    if 'description' in record_set:
        print(f"  Description: {record_set['description']}")
    else:
        print(f"  Description: (none)")
    print("  Fields:")
    for field in record_set['fields']:
        print(f"   - @id: {field['@id']}  (name: {field.get('name', '(no name)')})  (type: {field.get('dataType', '(unknown)')})")
    print("")

# Saving a main record_set @id for later use (if only one, take first)
main_record_set_id = record_sets[0]['@id'] if len(record_sets) > 0 else None
field_ids = [field['@id'] for field in record_sets[0]['fields']] if main_record_set_id else []

## 3. Data Extraction

Load data from the main record set into a DataFrame for analysis. Use the record set and field `@id` from the previous overview. The following code works with record set and field IDs and is robust to changes in the dataset structure.

In [ ]:
# Extract data from each record set (by @id)
dataframes = {}

for record_set in record_sets:
    rs_id = record_set['@id']
    records_list = list(dataset.records(record_set=rs_id))
    if len(records_list) > 0:
        dataframes[rs_id] = pd.DataFrame(records_list)
        print(f"Loaded {len(dataframes[rs_id])} records for record set {rs_id}.")
    else:
        print(f"No records found for record set {rs_id}.")

if main_record_set_id in dataframes:
    print(f"Columns in main record set ({main_record_set_id}):")
    print(dataframes[main_record_set_id].columns.tolist())
    dataframes[main_record_set_id].head()
else:
    print(f"Main record set {main_record_set_id} not loaded.")

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. You should adapt field selection depending on your exploration, always referencing fields by their `@id`.


In [ ]:
# To perform EDA, select a numeric field and, optionally, a group/categorical field. Use field @id as column names.
# We'll try to find a numeric field (like 'coverage_percent', 'peptide_count' or similar) and a group field ('accession', 'description' etc.)

# List columns for inspection
columns = dataframes[main_record_set_id].columns.tolist()
print("Available columns (by @id):\n", columns)

# For demonstration, pick the first column that is likely numeric
# Otherwise, set the field manually here as needed. Example: numeric_field_id = 'cr:coverage_percent'
import numpy as np

# Guess a numeric column
numeric_field = None
for col in columns:
    if dataframes[main_record_set_id][col].dtype in [float, int, np.float64, np.int64]:
        numeric_field = col
        break
if numeric_field is None:
    # Try to guess by name
    for col in columns:
        if 'coverage' in col or 'abundance' in col or 'count' in col or 'mw' in col.lower() or 'mass' in col.lower():
            numeric_field = col
            break

print(f"Using numeric field: {numeric_field}")

# Filter for values above a threshold
threshold = 10
df = dataframes[main_record_set_id]
if numeric_field is not None and numeric_field in df.columns:
    # Convert column to numeric, coercing errors
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold}:")
    print(filtered_df.head())

    # Normalize field
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try grouping by a likely categorical field
    group_field = None
    for cand in ['cr:description', 'cr:accession', 'cr:gene', 'description', 'accession', 'gene']:
        if cand in df.columns:
            group_field = cand
            break

    print(f"Using group field: {group_field}")

    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
        print(f"\nGrouped data by {group_field} (showing mean {numeric_field}):")
        print(grouped_df.head())
else:
    print("No numeric field found for EDA.")

## 5. Visualization

Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field is not None and numeric_field in df.columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()
else:
    print("No numeric field available for visualization.")

## 6. Conclusion

In this notebook, we used the `mlcroissant` library to explore the FAIR^2 dataset schema and retrieve records for analysis. All references to record sets and fields use their unique `@id`, promoting safe and reproducible code. We demonstrated basic exploratory data analysis on a numeric field, normalized the values, and visualized its distribution. You are encouraged to further explore other fields, group by attributes, or apply additional data science methods relevant to your research.